# 📘 pCN Chain Diagnostics — Theory & Intuition

This section explains the mathematical ideas behind MCMC diagnostics you will apply to your pCN chain:  
ESS, Trace Plots, Running Means, Autocorrelation, Posterior Mean/Variance, and Truth Comparison.

---

# ## 1. Effective Sample Size (ESS)

### **Definition**

For a Markov chain $$ u^{(1)}, u^{(2)}, \dots, u^{(M)} $$, samples are correlated.  
The **effective sample size** (ESS) measures how many *independent* samples the chain contains.

For one component $$u_k$$:

$$
\mathrm{ESS}_k = \frac{M}{\tau_k},
$$

where

$$
\tau_k = 1 + 2\sum_{\ell=1}^{\infty} \rho_k(\ell)
$$


---

### **Intuition**

- If mixing is good → autocorrelation is small → ESS ≈ M  
- If mixing is poor → autocorrelation is large → ESS ≪ M  
- ESS tells you **how much real information** your samples contain.

---

### **Rule of thumb**

- **ESS > 500** → excellent  
- **ESS > 100** → good  
- **ESS < 50** → bad  
- **ESS < 10** → unusable  

---

In [1]:
import numpy as np

data = np.load("./../../../Data/Experiment3/pcn_chain.npz")
chain = data["chain"]   # shape (num_samples, 64)
num_samples, dim = chain.shape

print("Chain shape:", chain.shape)

Chain shape: (1000000, 64)


In [3]:
chain=chain[-100000:,:]
print("Chain shape:", chain.shape)

Chain shape: (100000, 64)


In [2]:
import numpy as np

def autocorr_1d(x, max_lag=2000):
    """Compute autocorrelation for a 1D chain."""
    x = x - np.mean(x)
    n = len(x)
    result = np.correlate(x, x, mode='full')
    acf = result[result.size//2:] / result[result.size//2]
    return acf[:max_lag]

def ess_1d(x, max_lag=2000):
    """Effective sample size (ESS) for a 1D chain."""
    acf = autocorr_1d(x, max_lag=max_lag)
    positive_acf = acf[acf > 0]        # only sum positive lags
    tau = 1 + 2*np.sum(positive_acf[1:])
    return len(x) / tau

# ESS per dimension
ess = np.array([ess_1d(chain[:,i]) for i in range(dim)])

print("Minimum ESS across all 64 dims:", ess.min())
print("Median ESS:", np.median(ess))


In [ ]:
import matplotlib.pyplot as plt

idxs = [0, 10, 30]   # example components to inspect

plt.figure(figsize=(12,4))
for i, k in enumerate(idxs):
    plt.subplot(1,3,i+1)
    plt.plot(chain[:,k])
    plt.title(f"Trace plot for component {k}")
    plt.xlabel("Iteration")
    plt.ylabel("u[k]")
plt.tight_layout()
plt.show()
